In [1]:
# (Setup - you already have this in your notebook, skip it)
import pandas as pd
df = pd.read_excel("Project_Data.xlsx", keep_default_na=False, na_values=[""])

FileNotFoundError: [Errno 2] No such file or directory: 'Project_Data.xlsx'

## 1.3 Missing Values

The file is read with `keep_default_na=False`, so the text `"None"` (no insurance / no family history) stays as a normal category and is **not** counted as missing. Only the empty cells are real missing values.

### 1.3.1 Identifying Missing Values

In [ ]:
# number and percentage of missing values in each variable
missing = pd.DataFrame({"n_missing": df.isna().sum(),
                        "pct_missing": (df.isna().mean() * 100).round(2)})
print(missing[missing["n_missing"] > 0])

print("\nRows with at least one missing value:", df.isna().any(axis=1).sum())
print("Rows with more than one missing value:", (df.isna().sum(axis=1) > 1).sum())

# check if there are missing values hidden as text (for example "?" or "N/A")
hidden = ["", "?", "n/a", "na", "nan", "null", "unknown", "-"]
text_cols = df.select_dtypes(exclude=["number", "datetime"]).columns
n_hidden = 0
for col in text_cols:
    n_hidden = n_hidden + df[col].dropna().str.strip().str.lower().isin(hidden).sum()
print("Missing values hidden as text:", n_hidden)

Five variables have missing values: **Age**, **Education Level**, **Insurance Provider**, **Insurance Coverage** and **Approximate Annual Income**. They are all small (between 0.29% and 1.54% of the rows) and almost never in the same row (only 7 rows have more than one). No missing values are hidden as text.

Before choosing how to fix them, we check if the missing values are spread over many patients and if the patients still have the value in their other visits.

In [ ]:
cols_missing = missing[missing["n_missing"] > 0].index

# for each variable: how many patients are affected and how many have NO value in any visit
for col in cols_missing:
    patients_affected = df.loc[df[col].isna(), "Patient ID"].nunique()
    patients_no_value = (df.groupby("Patient ID")[col].count() == 0).sum()
    print(col, "| patients affected:", patients_affected, "| patients with no value at all:", patients_no_value)

# % of missing values by department
print()
display((df[cols_missing].isna().groupby(df["Department"]).mean() * 100).round(1))

**What we see**
- Almost every affected patient still has the value in other visits. The only exception is **Insurance Provider**: 2 patients have it missing in all their visits.
- Age is only missing in 5 departments and income in 6 other departments, so the missing values depend on the department that recorded the visit and not on the patient. It looks like a data collection problem and not something related to the patient's characteristics.
- Age, Education Level and Insurance Provider should not change between visits of the same patient, so the other visits of the patient give us the real value.

### 1.3.2 Handling Missing Values

| Variable | Missing rows | Decision | Why |
|---|---|---|---|
| Age, Education Level | 56 / 29 | Impute with the same patient's other visits | They are fixed per patient, so we do not invent anything |
| Insurance Provider | 104 | Impute with the same patient's other visits. For the 2 patients with no value, use coverage / price | Each provider pays a fixed share of the price, so the ratio shows the provider |
| Insurance Coverage | 50 | Impute with provider rate x price | Coverage follows this rule in the data |
| Approximate Annual Income | 154 | Impute with the patient's median income | It changes a bit between visits, so the median is better than the first value |

We do not remove any row or patient: there are few missing values and every patient still has information in other visits. We also do not use the average of all patients, because it would give a patient an age or income that is not theirs.

In [ ]:
# Age and Education Level: fixed for each patient, so we use the value of the other visits
for col in ["Age", "Education Level"]:
    df[col] = df[col].fillna(df.groupby("Patient ID")[col].transform("first"))

# Insurance Provider: first try the other visits of the same patient
df["Insurance Provider"] = df["Insurance Provider"].fillna(df.groupby("Patient ID")["Insurance Provider"].transform("first"))
print("Insurance Provider still missing:", df["Insurance Provider"].isna().sum(), "rows")
print("Patients with no provider in any visit:", df.loc[df["Insurance Provider"].isna(), "Patient ID"].unique())

# these patients have no provider at all, so we look at coverage / price for each provider
ratio = df["Insurance Coverage"] / df["Consultation Price"]
print("\nMedian of coverage / price by provider:")
print(ratio.groupby(df["Insurance Provider"]).median().round(2))

Each provider pays a fixed share of the price (None 0%, A 80%, B 90%, C 70%, D 100%), so the ratio identifies the provider. We use it for the two patients without provider and, in the next cell, to fill the missing coverage.

In [ ]:
rate_by_provider = {"None": 0.0, "Provider A": 0.8, "Provider B": 0.9, "Provider C": 0.7, "Provider D": 1.0}
provider_by_rate = {rate: provider for provider, rate in rate_by_provider.items()}

# Insurance Provider: median ratio of the patient, rounded, tells us the provider
patient_ratio = ratio.groupby(df["Patient ID"]).transform("median").round(1)
df["Insurance Provider"] = df["Insurance Provider"].fillna(patient_ratio.map(provider_by_rate))

# Insurance Coverage: provider rate x price
missing_cov = df["Insurance Coverage"].isna()
print("Missing coverage rows with price above 1000:", (missing_cov & (df["Consultation Price"] > 1000)).sum())
rate = df["Insurance Provider"].map(rate_by_provider)
df["Insurance Coverage"] = df["Insurance Coverage"].fillna(rate * df["Consultation Price"])

# Approximate Annual Income: changes between visits, so we use the median of the patient
df["Approximate Annual Income"] = df["Approximate Annual Income"].fillna(
    df.groupby("Patient ID")["Approximate Annual Income"].transform("median"))

# final check
print("\nMissing values after the imputation:")
print(df.isna().sum())
print("Total missing values:", df.isna().sum().sum())

All the missing values are filled and no row or patient was removed. Two small notes:
- None of the 50 missing coverage rows has an extreme price, so the filled coverage values are not affected by the price outliers (these are treated later in the outliers section).
- Many young patients have an income of 0. This is a valid value (not a missing one), so it is kept and the patient median keeps it at 0 for them.